# Simplified Pipeline — PM2.5 in Kampala and Nairobi

**By the end of this notebook, you will be able to:**
- Train a scikit-learn model using the `fit`/`predict` pattern shared by (almost) every model in the library
- Evaluate a model by testing it on a city it never trained on, instead of a random split
- Explain, concretely, why a raw geographic coordinate can break a linear model

Goal: build, end to end, the smallest complete and methodologically correct Machine Learning pipeline: load, clean, split, engineer features, train a model, evaluate. Each step is a block of code to complete. The train/test split principle here is deliberately simple: **train on one city, test on the other**.

In [72]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

TRAIN_CITY = "Kampala"
TEST_CITY = "Nairobi"
COLUMNS = [
    "city", "date", "hour", "site_latitude", "site_longitude", "pm2_5",
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]

## Step 1 — Load and restrict the scope

Load `data/train.csv`, keep only the two cities and the columns listed above. Then, so the pipeline stays fast to run and easy to inspect, cap the sample at 1200 rows per city.

**Documentation references:**
- [`Series.isin()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)
- [`DataFrame.sample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)

**Exercise:** Filter the raw data to the two cities and the listed columns. Then, city by city, draw a random sample of at most 1200 rows — pass a fixed `random_state` so the sample, and every metric computed from it later, stays reproducible between runs. Combine the two samples back into one table.

In [73]:
# TODO 1: load train.csv, filter the two cities and the columns, cap at 1200 rows/city
data= pd.read_csv("../../data/train.csv")
df= data.filter(COLUMNS)
df_filtered = df[df['city'].isin(["Kampala","Nairobi"])]
Nairobi_sample= (df_filtered[df_filtered['city']=='Nairobi']).sample(n=1200,random_state=42)
Kampala_sample= (df_filtered[df_filtered['city']=='Kampala']).sample(n=1200,random_state=42)

final_sample= pd.concat([Nairobi_sample,Kampala_sample],ignore_index=True)

## Step 2 — Clean missing values

Some satellite measurements are missing for some passes — you already measured how much, per city, in the discovery notebook. Fill them **within each city**, sorted by date: never let one city's values fill another city's gaps, that would be a geographic information leak.

**Documentation references:**
- [`DataFrameGroupBy.transform()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.transform.html)
- [`Series.ffill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.ffill.html) / [`Series.bfill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.bfill.html)

**Exercise:** Sort the rows by city and date. Then, for each fillable column, group by city and use `transform` to apply forward-fill followed by backward-fill *within* each group — `transform` is what lets you get a filled column back in the original row order, ready to reassign to `df`, rather than one small table per city to reassemble yourself.

In [74]:
# TODO 2: fill missing values by city, sorted by date
final_sample = final_sample.sort_values(by=['city','date']).reset_index(drop=True)
ruined_columns= ["sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",]

for column in ruined_columns:
    final_sample[column]= final_sample.groupby('city')[column].transform(lambda x: x.ffill().bfill())

## Step 3 — Manual split by city

A basic way to check whether a model actually generalizes is to test it on data it has never seen. Here, the simplest version of that idea: train on `TRAIN_CITY`, evaluate on `TEST_CITY` — no random shuffling of rows. That is enough to get started in this activity. Later on, you will see a more rigorous and systematic way to do this across several groups at once, and to guard against overfitting — cross-validation, covered in a dedicated Session 3 module.

**Exercise:** Split `df` into `train_df` (rows where `city == TRAIN_CITY`) and `test_df` (rows where `city == TEST_CITY`).

In [75]:
# TODO 3: split into train_df (TRAIN_CITY) and test_df (TEST_CITY)
train_df = final_sample[final_sample['city'] == TRAIN_CITY]
test_df = final_sample[final_sample['city'] == TEST_CITY]

## Step 4 — Temporal features

A model can only exploit the patterns it can actually see in its input. A raw date doesn't expose seasonality or a weekly rhythm — nothing in a linear model lets it notice "this is a rainy month" or "this is a weekday" just by looking at a timestamp. When you know, from domain knowledge, that a pattern is likely to matter — here, pollution following seasonal and weekly cycles from traffic and economic activity — it is worth engineering it explicitly as a feature rather than hoping the model finds it on its own. That is why the next step extracts the month and the day of week from `date`.

**Exercise:** For both `train_df` and `test_df`, add `month` and `dayofweek` columns from `date`, using the same `.dt` accessor you already used in the discovery notebook.

In [76]:
# TODO 4: add month and dayofweek columns derived from date
train_df['date']= pd.to_datetime(train_df['date'])
test_df['date']= pd.to_datetime(test_df['date'])

train_df['dayofweek']= train_df['date'].dt.day
test_df['dayofweek']= test_df['date'].dt.day

train_df['month']= train_df['date'].dt.month
test_df['month']= test_df['date'].dt.month

## Step 5 — Baseline model and evaluation

Every scikit-learn model follows the same two-step contract, no matter how simple or advanced: `model.fit(X_train, y_train)` learns from data, `model.predict(X_test)` applies what it learned to new data. You will reuse this exact pattern for every model in this course, starting with the simplest one, `LinearRegression`.

**Warning**: `site_latitude`/`site_longitude` are deliberately excluded from the features. In the training sample, one city only occupies a small geographic area (a handful of very close monitoring sites): a linear model fits a huge coefficient on these coordinates, which explodes as soon as it is applied to very different coordinates (the other city). Try adding them back to `feature_cols` to see the effect on the metrics.

For a refresher on what RMSE, MAE, and R² actually measure, see [section 6.2 of Machine Learning Techniques: From Theory to Practice](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#62-evaluation-metrics-for-regression).

**Documentation references:**
- [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [`mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [`mean_absolute_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), [`r2_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

**Exercise:** Build `feature_cols`: every numeric column of `train_df` except `pm2_5` (the target) and the columns named in the warning above. Fit a `LinearRegression` on `train_df`, predict on `test_df`, then compute RMSE (the square root of `mean_squared_error`), MAE, and R².

In [86]:
# TODO 5: define the features, train the linear model, compute RMSE/MAE/R2
feature_col = [
    "site_latitude", "site_longitude",
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]
feature_col_cleaned = [
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]
L=[feature_col,feature_col_cleaned]
for features in L:
    x_train= train_df[features]
    x_test= test_df[features]
    y_test= test_df['pm2_5']

    target= train_df['pm2_5']
    model= LinearRegression()
    model.fit(x_train,target)
    y_predicted = model.predict(x_test)

    error_1= mean_squared_error(y_test,y_predicted)
    error_2= mean_absolute_error(y_test,y_predicted)
    error_3= r2_score(y_test,y_predicted)
    print(str(features))
    display(error_1)
    display(error_2)
    display(error_3)

['site_latitude', 'site_longitude', 'sulphurdioxide_so2_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_no2_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density', 'uvaerosolindex_absorbing_aerosol_index', 'ozone_o3_column_number_density', 'uvaerosollayerheight_aerosol_height', 'cloud_cloud_fraction']


113302.47237724706

335.6754486438629

-190.4068556995848

['sulphurdioxide_so2_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_no2_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density', 'uvaerosolindex_absorbing_aerosol_index', 'ozone_o3_column_number_density', 'uvaerosollayerheight_aerosol_height', 'cloud_cloud_fraction']


611.7657700726776

10.086930012791123

-0.03348285361653902

## Going further

What would happen if you reversed it: train on Nairobi and test on Kampala? Try it in the cell below and compare the two errors. What does this tell you about how hard it is to generalize from one city to another?

In Session 3, you will see how to automate this kind of comparison across more than two cities at once with `GroupKFold` — see [section 4.2.3](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#423-groupkfold-cross-validation) for a preview.

In [87]:
"paste everything"
# TODO 3: split into train_df (TRAIN_CITY) and test_df (TEST_CITY)
train_df = final_sample[final_sample['city'] == TEST_CITY]
test_df = final_sample[final_sample['city'] == TRAIN_CITY]
# TODO 4: add month and dayofweek columns derived from date
train_df['date']= pd.to_datetime(train_df['date'])
test_df['date']= pd.to_datetime(test_df['date'])

train_df['dayofweek']= train_df['date'].dt.day
test_df['dayofweek']= test_df['date'].dt.day

train_df['month']= train_df['date'].dt.month
test_df['month']= test_df['date'].dt.month
# TODO 5: define the features, train the linear model, compute RMSE/MAE/R2
feature_col = [
    "site_latitude", "site_longitude",
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]
feature_col_cleaned = [
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]
L=[feature_col,feature_col_cleaned]
for features in L:
    x_train= train_df[features]
    x_test= test_df[features]
    y_test= test_df['pm2_5']

    target= train_df['pm2_5']
    model= LinearRegression()
    model.fit(x_train,target)
    y_predicted = model.predict(x_test)

    error_1= mean_squared_error(y_test,y_predicted)
    error_2= mean_absolute_error(y_test,y_predicted)
    error_3= r2_score(y_test,y_predicted)
    print(str(features))
    display(error_1)
    display(error_2)
    display(error_3)

['site_latitude', 'site_longitude', 'sulphurdioxide_so2_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_no2_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density', 'uvaerosolindex_absorbing_aerosol_index', 'ozone_o3_column_number_density', 'uvaerosollayerheight_aerosol_height', 'cloud_cloud_fraction']


27484.32621267422

165.02389502469978

-137.32464051832588

['sulphurdioxide_so2_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_no2_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density', 'uvaerosolindex_absorbing_aerosol_index', 'ozone_o3_column_number_density', 'uvaerosollayerheight_aerosol_height', 'cloud_cloud_fraction']


244.9875235988128

11.906006420259788

-0.23298678930879202